<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Compare Active and Inactive Bounds

**Change only the correction limit and observe which KKT case survives.**

Keep the clock observations, squared-error objective, and exact calibration. The upper bound determines whether the equality-only minimum is still available.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Retain calibration and impose an upper bound

The observed errors are $d_i=-2,-1,-1,0$ min at $t_i=0,1,2,3$ h, for $i=1,\ldots,4$. Choose initial correction $q$ in min and rate correction $r$ in min/h. The remaining errors are $y_i=d_i+q+rt_i$ and the squared-error objective is

$$
\phi(q,r)=\sum_{i=1}^{4}y_i^2.
$$

The original calibration requirement at one hour is $q+(1\,\mathrm h)r=1\,\mathrm{min}$. Using numerical values in these units, keep the equality residual $h(q,r)=q+r-1=0$. Let $q_{\max}$ be the upper limit on the initial correction, in minutes, and define the inequality residual $g(q,r)=q-q_{\max}\le0$.

Unless a comparison states otherwise, use $q_{\max}=1$. The equality-only solution $(1.5,-0.5)$ violates this upper bound. The next definition keeps physical errors, the objective, and feasibility checks separate.


In [ ]:
import numpy as np

# Fixed observations and their elapsed times
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h

# Calibration requirement: correction at 1 h must equal 1 min
CALIBRATION_TIME = 1.0        # h
REQUIRED_CORRECTION = 1.0     # min
CHECK_TOLERANCE = 1e-9        # numerical comparison setting


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    remaining_errors = simulate_clock(decision)
    correction, rate = decision
    equality_residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    gradient = 2 * np.array([np.sum(remaining_errors),
                             np.dot(OBSERVATION_TIMES, remaining_errors)])
    return {
        "decision": decision,
        "response": remaining_errors,
        "objective": float(np.sum(remaining_errors ** 2)),
        "gradient": gradient,
        "equality_residual": float(equality_residual),
        "feasible": bool(abs(equality_residual) <= CHECK_TOLERANCE),
    }


CORRECTION_LIMIT = 1.0  # min


def evaluate_bounded_clock(decision, correction_limit=CORRECTION_LIMIT):
    result = evaluate_clock(decision)
    inequality_residual = float(result["decision"][0] - correction_limit)
    result.update({
        "inequality_residual": inequality_residual,
        "feasible": bool(result["feasible"] and inequality_residual <= CHECK_TOLERANCE),
    })
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Use two simple cases

Let $\nu$ be the equality multiplier and $\mu\ge0$ the inequality multiplier. The Lagrangian is $L=\phi+\nu h+\mu g$. Complementarity, $\mu g=0$, gives two cases:

| Case | Candidate | Required check |
|:---|:---|:---|
| Set $\mu=0$ | The equality-only settings $(1.5,-0.5)$ | They must fit under the bound |
| Set $q=q_{\max}$ | Calibration gives $r=1-q_{\max}$ | The resulting multiplier must satisfy $\mu\ge0$ |

An **active set** lists the inequalities that hold exactly at their limits. With one bound, the set either contains that bound or is empty. At a bound with a zero multiplier, both case descriptions can give the same candidate.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Run the tight and loose cases

For $q_{\max}=1$, the equality-only correction 1.5 is rejected. The active candidate $(1,0)$ survives, with $\nu=-6$ and $\mu=6$.

For $q_{\max}=2$, the equality-only settings fit and have $\mu=0$. Forcing the boundary setting $(2,-1)$ would require $\mu=-6$, so that case is rejected.

Along calibration, the score decreases toward $q=1.5$ and increases after it. The surviving correction is therefore $q^\star=\min(1.5,q_{\max})$, with $r^\star=1-q^\star$. The function implements this clock-specific case calculation and obtains the multipliers from the two slope balances.


In [ ]:
def solve_bounded_clock(correction_limit):
    correction = min(1.5, float(correction_limit))
    rate = REQUIRED_CORRECTION - correction
    result = evaluate_bounded_clock([correction, rate], correction_limit)
    q_slope, r_slope = result["gradient"]
    equality_multiplier = -r_slope
    inequality_multiplier = -q_slope - equality_multiplier
    result.update({
        "equality_multiplier": float(equality_multiplier),
        "inequality_multiplier": float(inequality_multiplier),
        "active": bool(abs(result["inequality_residual"]) <= CHECK_TOLERANCE),
    })
    return result


for limit in (1.0, 2.0):
    result = solve_bounded_clock(limit)
    print(f"Limit {limit:.1f} min | settings {result['decision']}"
          f" | score {result['objective']:.2f} min²"
          f" | nu {result['equality_multiplier']:.2f}"
          f" | mu {result['inequality_multiplier']:.2f}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

At the intermediate limit 1.5 min, the optimum lies exactly on the bound and has $\mu=0$. The code checks this boundary case along with the tight and loose cases. All three selected candidates pass every KKT check.


In [ ]:
def check_kkt(result):
    equality_multiplier = result["equality_multiplier"]
    inequality_multiplier = result["inequality_multiplier"]
    stationarity = (result["gradient"]
                    + equality_multiplier * np.array([1.0, CALIBRATION_TIME])
                    + inequality_multiplier * np.array([1.0, 0.0]))
    complementarity = inequality_multiplier * result["inequality_residual"]
    return {
        "feasibility": result["feasible"],
        "multiplier_sign": bool(inequality_multiplier >= -CHECK_TOLERANCE),
        "complementarity": bool(abs(complementarity) <= CHECK_TOLERANCE),
        "stationarity": bool(np.all(np.abs(stationarity) <= CHECK_TOLERANCE)),
    }


for limit in (1.0, 1.5, 2.0):
    result = solve_bounded_clock(limit)
    print(f"Limit {limit:.1f} min | active = {result['active']} | mu = {result['inequality_multiplier']:.2f}")
    print(check_kkt(result))


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · See when the bound changes the selected setting

Compare the orange points while only the limit changes. A tight limit moves the selected setting to its boundary. Once the equality-only minimum fits, raising the limit further leaves the selected setting unchanged.
<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/bound_activity_and_multiplier.png" alt="With calibration retained, correction limits 1, 1.5, and 2 select corrections 1, 1.5, and 1.5. The corresponding inequality multipliers are 6, 0, and 0." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The executable plot uses the same evaluation function. Teal shading marks allowed corrections within the view; the orange point is the selected setting. The displayed axis range does not add another constraint.


In [ ]:
import matplotlib.pyplot as plt


def show_bound_analysis(correction_limit=1.0):
    with plt.rc_context({"font.size": 11.52}):
        figure, axis = plt.subplots(figsize=(5.12, 3.2), dpi=100)
        corrections = np.linspace(0.3, 2.1, 181)
        scores = [evaluate_clock([q, 1 - q])["objective"] for q in corrections]
        axis.plot(corrections, scores, color="#286d9b")
        axis.axvspan(0.3, min(correction_limit, 2.1), color="#177b70", alpha=0.12)
        axis.axvline(correction_limit, color="#177b70", linestyle="--")
        selected = solve_bounded_clock(correction_limit)
        axis.scatter(selected["decision"][0], selected["objective"],
                     color="#b86522", zorder=4)
        axis.set(xlim=(0.3, 2.1), ylim=(0, 10),
                 xlabel="Correction q (min)", ylabel="Score (min²)",
                 title=("A tight bound blocks the lower score"
                        if correction_limit < 1.5 else "A loose bound leaves the minimum available"))
        axis.set_xticks([0.5, 1.0, 1.5, 2.0])
        axis.set_yticks([0, 4, 8])
        axis.grid(alpha=0.25)
        figure.tight_layout()
        return figure


In [ ]:
bound_figure = show_bound_analysis(correction_limit=1.0)
plt.show()
